In [1]:
from pathlib import Path
from io import BytesIO
from collections import Counter, defaultdict
import random
import time

import albumentations as A
import cv2
import numpy as np
import pyarrow as pa

# Manipulation des tables Parquet et Arrow.

from pyarrow import fs
import pyarrow.parquet as pq
from PIL import Image
import pandas as pd

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Crée l'objet PyArrow qui communique avec Amazon S3.
s3 = fs.S3FileSystem(region="us-east-1")

# Nom du bucket S3.
BUCKET_NAME = "mushroom-ml-mikael-2026-000"

# Fichier source contenant les données nettoyées et les ID.
PARQUET_PATH = (
    f"{BUCKET_NAME}/processed/parquet/train.parquet"

)

OUTPUT_PATH = (
    f"{BUCKET_NAME}/processed/parquet/train_balanced_augmented.parquet"

)


table = pq.read_table(PARQUET_PATH,filesystem=s3)



print("Lignes :", table.num_rows)
print("Colonnes :", table.column_names)

Lignes : 1705
Colonnes : ['image', 'label', 'bbox', 'origin', 'mushroom_name', 'edibility', 'entry_id']


In [2]:
class_counts = Counter(
    table["mushroom_name"].to_pylist()
)

class_counts

Counter({'Entoloma_sinuatum': 204,
         'Rubroboletus_satanas': 199,
         'Hygrophoropsis_aurantiaca': 173,
         'Agaricus_bisporus': 167,
         'Boletus_edulis': 149,
         'Armillaria_mellea': 147,
         'Hypholoma_fasciculare': 131,
         'Pleurotus_ostreatus': 131,
         'Clitocybe_dealbata': 125,
         'Lactarius_resimus': 120,
         'Amanita_phalloides': 98,
         'Cantharellus_cibarius': 61})

In [3]:
#Adding columns unique entry_id to the table for easier tracking of each row
#entry_ids = [
#    f"{i:06d}"
#    for i in range(table.num_rows)
#]
#
#table = table.append_column(
#    "entry_id",
#    pa.array(entry_ids)
#)

table.to_pandas().head(10)
table.select(["entry_id", "mushroom_name"]).slice(0, 10).to_pandas()

,image,label,bbox,origin,mushroom_name,edibility,entry_id
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Agaricus_bisporus_edible,"[85.0, 67.0, 519.0, 529.5]",natural,Agaricus_bisporus,edible,002208
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Hypholoma_fasciculare_poisonous,"[0.0, 34.0, 625.0, 639.5]",natural,Hypholoma_fasciculare,poisonous,001323
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Hypholoma_fasciculare_poisonous,"[51.0, 321.0, 231.5, 591.5]",natural,Hypholoma_fasciculare,poisonous,001374
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Hypholoma_fasciculare_poisonous,"[141.0, 14.0, 308.0, 233.778]",natural,Hypholoma_fasciculare,poisonous,002149
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Agaricus_bisporus_edible,"[97.0, 120.0, 468.0, 503.0]",natural,Agaricus_bisporus,edible,000488
5,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Armillaria_mellea_edible,"[136.0, 138.0, 310.0, 260.0]",natural,Armillaria_mellea,edible,000569
6,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Agaricus_bisporus_edible,"[201.0, 236.0, 594.0, 570.0]",natural,Agaricus_bisporus,edible,001603
7,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Entoloma_sinuatum_poisonous,"[327.0, 68.0, 487.0, 605.0]",natural,Entoloma_sinuatum,poisonous,001108
8,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Hygrophoropsis_aurantiaca_poisonous,"[194.0, 88.0, 290.641, 188.643]",natural,Hygrophoropsis_aurantiaca,poisonous,001232
9,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Hypholoma_fasciculare_poisonous,"[28.0, 146.0, 218.0, 478.0]",natural,Hypholoma_fasciculare,poisonous,001401


,entry_id,mushroom_name
0,002208,Agaricus_bisporus
1,001323,Hypholoma_fasciculare
2,001374,Hypholoma_fasciculare
3,002149,Hypholoma_fasciculare
4,000488,Agaricus_bisporus
5,000569,Armillaria_mellea
6,001603,Agaricus_bisporus
7,001108,Entoloma_sinuatum
8,001232,Hygrophoropsis_aurantiaca
9,001401,Hypholoma_fasciculare


In [4]:
#Adding columns telling if the entry is generated or not (natural / generated)

#origin = pa.array(
#    ["natural"] * table.num_rows
#)
#
#table = table.append_column(
#    "origin",
#    origin
#)

In [5]:
class_counts_df = pd.DataFrame(
    class_counts.items(),
    columns=["mushroom_name", "nombre"]
).sort_values(
    "nombre",
    ascending=False
)

class_counts_df

target_count = max(class_counts.values())

print("Taille cible :", target_count)

,mushroom_name,nombre
3,Entoloma_sinuatum,204
6,Rubroboletus_satanas,199
4,Hygrophoropsis_aurantiaca,173
0,Agaricus_bisporus,167
8,Boletus_edulis,149
2,Armillaria_mellea,147
1,Hypholoma_fasciculare,131
9,Pleurotus_ostreatus,131
5,Clitocybe_dealbata,125
11,Lactarius_resimus,120


Taille cible : 204


In [6]:
augmentation_plan = class_counts_df.copy()

augmentation_plan["a_creer"] = (
    target_count - augmentation_plan["nombre"]
)

augmentation_plan

,mushroom_name,nombre,a_creer
3,Entoloma_sinuatum,204,0
6,Rubroboletus_satanas,199,5
4,Hygrophoropsis_aurantiaca,173,31
0,Agaricus_bisporus,167,37
8,Boletus_edulis,149,55
2,Armillaria_mellea,147,57
1,Hypholoma_fasciculare,131,73
9,Pleurotus_ostreatus,131,73
5,Clitocybe_dealbata,125,79
11,Lactarius_resimus,120,84


In [7]:
# Define the augmentation pipeline applied to training images.
transform = A.Compose([

    # Randomly flip the image horizontally.
    A.HorizontalFlip(p=0.5),

    # Apply random scaling, translation, rotation and shear.
    A.Affine(
        scale=(0.90, 1.10),
        translate_percent=(-0.05, 0.05),
        rotate=(-20, 20),
        shear=(-5, 5),
        p=0.7
    ),

    # Simulate a slight camera viewpoint change.
    A.Perspective(
        scale=(0.02, 0.05),
        p=0.2
    ),

    # Randomly adjust brightness and contrast.
    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.4
    ),

    # Apply small color variations.
    A.HueSaturationValue(
        hue_shift_limit=5,
        sat_shift_limit=10,
        val_shift_limit=10,
        p=0.25
    ),

    # Randomly apply either blur or Gaussian noise.
    A.OneOf([
        A.GaussianBlur(
            blur_limit=(3, 5),
            p=1.0
        ),
        A.GaussNoise(
            std_range=(0.01, 0.04),
            p=1.0
        )
    ], p=0.15),

    # Simulate image compression artifacts.
    A.ImageCompression(
        quality_range=(80, 100),
        p=0.15
    )
    ],
    
                      
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["bbox_labels"],
        min_visibility=0.80)
    ,

    seed=42
)

In [8]:
def decode_image(image_data):
    image_bytes = image_data["bytes"]

    image = Image.open(
        BytesIO(image_bytes)
    ).convert("RGB")

    return np.array(image)


def encode_image(image_array):
    buffer = BytesIO()

    Image.fromarray(image_array).save(
        buffer,
        format="JPEG",
        quality=95
    )

    return buffer.getvalue()

In [9]:
def augment_row(source_row, artificial_entry_id):
    image = decode_image(source_row["image"])

    height, width = image.shape[:2]

    x_min, y_min, x_max, y_max = [
        float(value)
        for value in source_row["bbox"]
    ]

    # Empêche la bbox de dépasser les limites de l'image
    x_min = max(0.0, min(x_min, float(width)))
    y_min = max(0.0, min(y_min, float(height)))
    x_max = max(0.0, min(x_max, float(width)))
    y_max = max(0.0, min(y_max, float(height)))

    # Refuse une bbox devenue vide ou incorrecte
    if x_max <= x_min or y_max <= y_min:
        return None

    bbox = [
        x_min,
        y_min,
        x_max,
        y_max
    ]

    result = transform(
        image=image,
        bboxes=[bbox],
        bbox_labels=[source_row["mushroom_name"]]
    )

    # La bbox peut disparaître après une transformation géométrique
    if len(result["bboxes"]) == 0:
        return None

    augmented_image = result["image"]
    augmented_bbox = result["bboxes"][0]

    new_row = source_row.copy()
    new_image_data = dict(source_row["image"])

    new_image_data["bytes"] = encode_image(
        augmented_image
    )

    if "path" in new_image_data:
        new_image_data["path"] = None

    new_row["image"] = new_image_data

    new_row["bbox"] = [
        float(value)
        for value in augmented_bbox
    ]

    new_row["entry_id"] = artificial_entry_id
    new_row["origin"] = "artificial"

    return new_row

In [10]:
natural_rows = [
    row
    for row in table.to_pylist()
    if row["origin"] == "natural"
]

rows_by_class = defaultdict(list)

for row in natural_rows:
    rows_by_class[
        row["mushroom_name"]
    ].append(row)


{
    mushroom_name: len(rows)
    for mushroom_name, rows
    in rows_by_class.items()
}

{'Agaricus_bisporus': 167,
 'Hypholoma_fasciculare': 131,
 'Armillaria_mellea': 147,
 'Entoloma_sinuatum': 204,
 'Hygrophoropsis_aurantiaca': 173,
 'Clitocybe_dealbata': 125,
 'Rubroboletus_satanas': 199,
 'Amanita_phalloides': 98,
 'Boletus_edulis': 149,
 'Pleurotus_ostreatus': 131,
 'Cantharellus_cibarius': 61,
 'Lactarius_resimus': 120}

In [11]:
augmentation_counters = defaultdict(int)
augmented_rows = []

random.seed(42)


for mushroom_name, class_rows in rows_by_class.items():

    current_count = len(class_rows)
    needed = target_count - current_count

    print(
        f"{mushroom_name}: "
        f"{current_count} naturelles, "
        f"{needed} artificielles à créer"
    )

    created = 0
    attempts = 0

    while created < needed:
        attempts += 1

        source_row = random.choice(
            class_rows
        )

        parent_id = source_row["entry_id"]

        augmentation_counters[parent_id] += 1

        augmentation_number = (
            augmentation_counters[parent_id]
        )

        artificial_entry_id = (
            f"{parent_id}_"
            f"{augmentation_number:02d}"
        )

        new_row = augment_row(
            source_row,
            artificial_entry_id
        )

        if new_row is not None:
            augmented_rows.append(new_row)
            created += 1

    print(
        f"  → {created} images créées "
        f"en {attempts} tentatives"
    )

Agaricus_bisporus: 167 naturelles, 37 artificielles à créer


  → 37 images créées en 39 tentatives
Hypholoma_fasciculare: 131 naturelles, 73 artificielles à créer


  → 73 images créées en 78 tentatives
Armillaria_mellea: 147 naturelles, 57 artificielles à créer


  → 57 images créées en 63 tentatives
Entoloma_sinuatum: 204 naturelles, 0 artificielles à créer
  → 0 images créées en 0 tentatives
Hygrophoropsis_aurantiaca: 173 naturelles, 31 artificielles à créer


  → 31 images créées en 33 tentatives
Clitocybe_dealbata: 125 naturelles, 79 artificielles à créer


...

  → 106 images créées en 106 tentatives
Boletus_edulis: 149 naturelles, 55 artificielles à créer


  → 55 images créées en 63 tentatives
Pleurotus_ostreatus: 131 naturelles, 73 artificielles à créer


  → 73 images créées en 78 tentatives
Cantharellus_cibarius: 61 naturelles, 143 artificielles à créer


  → 143 images créées en 179 tentatives
Lactarius_resimus: 120 naturelles, 84 artificielles à créer


  → 84 images créées en 98 tentatives


In [12]:
augmented_table = pa.Table.from_pylist(
    augmented_rows, 
    schema=table.schema
)

balanced_table = pa.concat_tables([
    table,
    augmented_table
])

In [13]:
balanced_counts = (
    balanced_table
    .select(["mushroom_name"])
    .to_pandas()["mushroom_name"]
    .value_counts()
)

# Remove large Python objects.
del table
del augmented_table
del augmented_rows

balanced_counts

mushroom_name
Agaricus_bisporus            204
Hypholoma_fasciculare        204
Armillaria_mellea            204
Entoloma_sinuatum            204
Hygrophoropsis_aurantiaca    204
Clitocybe_dealbata           204
Rubroboletus_satanas         204
Amanita_phalloides           204
Boletus_edulis               204
Pleurotus_ostreatus          204
Cantharellus_cibarius        204
Lactarius_resimus            204
Name: count, dtype: int64

In [14]:
start_time = time.time()

pq.write_table(
    balanced_table,
    OUTPUT_PATH,
    compression="gzip",
    #row_group_size=128, #t3 large remove the need of grouping
    filesystem=s3
)
elapsed_time = time.time() - start_time

print(f"Export completed in {elapsed_time:.1f} seconds")


print("Table exportée vers :")
print(OUTPUT_PATH)

Export completed in 11.9 seconds
Table exportée vers :
mushroom-ml-mikael-2026-000/processed/parquet/train_balanced_augmented.parquet
